In [16]:
import pandas as pd
import re
import nltk
from gensim.corpora import Dictionary
from gensim.models import LdaModel
from nltk.stem import PorterStemmer


from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


### Get all unique values for spell.csv the "type coulmn"

In [17]:
import pandas as pd
import re
from nltk.tokenize import word_tokenize



df = pd.read_csv("Spells.csv")
print(df.head())
print(df.columns.tolist())

   Spell ID       Incantation                        Spell Name  \
0         1             Accio                   Summoning Charm   
1         2         Aguamenti                Water-Making Spell   
2         3  Alarte Ascendare  Launch an object up into the air   
3         4         Alohomora                   Unlocking Charm   
4         5     Arania Exumai            Spider repelling spell   

                  Effect     Light  
0      Summons an object       NaN  
1         Conjures water  Icy blue  
2  Rockets target upward       Red  
3         Unlocks target      Blue  
4         Repels spiders      Blue  
['Spell ID', 'Incantation', 'Spell Name', 'Effect', 'Light']


# Show all unique spell names

In [18]:
# Show all unique spell names
unique_spell_names = df["Spell Name"].dropna().unique()

print(unique_spell_names)
print(f"\nTotal unique spell names: {len(unique_spell_names)}")

['Summoning Charm' 'Water-Making Spell' 'Launch an object up into the air'
 'Unlocking Charm' 'Spider repelling spell' 'Slowing Charm'
 'Killing Curse' 'Exploding Charm' 'Brackium Emendo' 'Cistem Aperio'
 'Locking Spell' 'Blasting Curse' 'Cruciatus Curse' 'Severing Charm'
 'Dissendium' 'Engorgement Charm' 'Episkey' 'Patronus Charm'
 'Disarming Charm' 'Expulso Curse' 'General Counter-Spell'
 'Human Presence Revealing Spell' 'Freezing Charm' 'Impediment Jinx'
 'Imperius Curse' 'Incarcerous Spell' 'Fire-Making Spell' 'Levicorpus'
 'Locomotion Charm' 'Leg-Locker Curse' 'Wand-Lighting Charm'
 'Lumos Maxima' 'Lumos Solem Spell' 'Muffliato Charm'
 'Wand-Extinguishing Charm' 'Memory Charm' 'Oculus Reparo' 'Oppugno Jinx'
 'Peskipiksi Pesternomi' 'Full Body-Bind Curse' 'Piertotum Locomotor'
 'Portus' 'Reverse Spell' 'Shield Charm' 'Protego Maxima'
 'Protego totalum' 'Shrinking Charm' 'Revulsion Jinx' 'Mending Charm'
 'Repello Inimicum' 'Muggle-Repelling Charm' 'Revelio Charm'
 'Tickling Charm' '

In [19]:
print(df.isnull().sum())

Spell ID        0
Incantation     0
Spell Name      0
Effect          0
Light          21
dtype: int64


In [20]:
# Combine text fields
df["text"] = df["Spell Name"] + " " + df["Effect"]

# Lowercase
df["text"] = df["text"].str.lower()

# Remove punctuation
df["text"] = df["text"].apply(
    lambda x: re.sub(r"[^a-zA-Z\s]", "", x)
)
import nltk

# Tokenization
df["tokens"] = df["text"].apply(word_tokenize)

# Standard English stopwords
stop_words = set(stopwords.words("english"))

# Harry Potter / dataset-specific stopwords
custom_stopwords = {
    "spell",
    "charm",
    "curse",
    "jinx",
    "hex",
    "target",
    "object",
    "force",
    "forc",
    "maximum",
    "maxima",
    "wand",
    "turn",
    "injury",
    "injuries"
}

# Combine both sets
all_stopwords = stop_words.union(custom_stopwords)

# Remove stopwords
df["tokens"] = df["tokens"].apply(
    lambda words: [w for w in words if w not in all_stopwords]
)

stemmer = PorterStemmer()
lemmer = WordNetLemmatizer()

df["tokens"] = df["tokens"].apply(
    #lambda words: [stemmer.stem(w) for w in words]
    lambda words: [lemmer.lemmatize(w) for w in words]
)

# View results
print(df[["Spell Name", "tokens"]].head(20))

                          Spell Name                                    tokens
0                    Summoning Charm                      [summoning, summons]
1                 Water-Making Spell            [watermaking, conjures, water]
2   Launch an object up into the air             [launch, air, rocket, upward]
3                    Unlocking Charm                      [unlocking, unlocks]
4             Spider repelling spell       [spider, repelling, repels, spider]
5                      Slowing Charm  [slowing, slows, stop, target, velocity]
6                      Killing Curse           [killing, instantaneous, death]
7                    Exploding Charm             [exploding, small, explosion]
8                    Brackium Emendo            [brackium, emendo, mend, bone]
9                      Cistem Aperio             [cistem, aperio, open, chest]
10                     Locking Spell                     [locking, lock, door]
11                    Blasting Curse                

### Making a dictionary

In [21]:
dictionary = Dictionary(df["tokens"])
dictionary.filter_extremes(
    no_below=2,
    no_above=0.7
)
corpus = [dictionary.doc2bow(text) for text in df["tokens"]]

### Small model

In [22]:
lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=4,
    random_state=42,
    passes=20
)

In [23]:
for idx, topic in lda_model.print_topics():
    print(f"Topic {idx}: {topic}")

Topic 0: 0.260*"reveals" + 0.180*"secret" + 0.180*"turn" + 0.099*"summons" + 0.020*"stop" + 0.020*"water" + 0.020*"protego" + 0.020*"spell" + 0.020*"object" + 0.020*"explosion"
Topic 1: 0.318*"object" + 0.136*"produce" + 0.136*"lumos" + 0.076*"summons" + 0.076*"protego" + 0.075*"water" + 0.015*"snake" + 0.015*"spell" + 0.015*"explosion" + 0.015*"conjures"
Topic 2: 0.210*"repels" + 0.145*"movement" + 0.145*"explosion" + 0.145*"stop" + 0.081*"protego" + 0.080*"spell" + 0.016*"object" + 0.016*"summons" + 0.016*"water" + 0.016*"force"
Topic 3: 0.218*"conjures" + 0.166*"snake" + 0.115*"shield" + 0.115*"force" + 0.065*"spell" + 0.065*"summons" + 0.065*"water" + 0.064*"protego" + 0.013*"repels" + 0.013*"object"


In [24]:
# Store dominant topic for each spell
dominant_topics = []

# Store full topic probabilities
topic_distributions = []

for bow in corpus:

    # Get topic probabilities
    topic_probs = lda_model.get_document_topics(bow)

    # Save full probability distribution
    topic_distributions.append(topic_probs)

    # Get dominant topic
    dominant_topic = max(topic_probs, key=lambda x: x[1])

    dominant_topics.append(dominant_topic[0])

# Add dominant topic column to dataframe
df["Dominant Topic"] = dominant_topics
print(df[["Spell Name", "Dominant Topic"]].head(30))

                          Spell Name  Dominant Topic
0                    Summoning Charm               0
1                 Water-Making Spell               3
2   Launch an object up into the air               0
3                    Unlocking Charm               0
4             Spider repelling spell               2
5                      Slowing Charm               2
6                      Killing Curse               0
7                    Exploding Charm               2
8                    Brackium Emendo               0
9                      Cistem Aperio               0
10                     Locking Spell               0
11                    Blasting Curse               2
12                   Cruciatus Curse               0
13                    Severing Charm               1
14                        Dissendium               0
15                 Engorgement Charm               0
16                           Episkey               0
17                    Patronus Charm          

In [25]:
topic_labels = {
    0: "Combat/Force Magic",
    1: "Protective Utility Magic",
    2: "Revealing/Conjuration Magic"
}

df["Topic Label"] = df["Dominant Topic"].map(topic_labels)

In [26]:
print(df[["Spell Name", "Dominant Topic", "Topic Label"]].head(20))

                          Spell Name  Dominant Topic  \
0                    Summoning Charm               0   
1                 Water-Making Spell               3   
2   Launch an object up into the air               0   
3                    Unlocking Charm               0   
4             Spider repelling spell               2   
5                      Slowing Charm               2   
6                      Killing Curse               0   
7                    Exploding Charm               2   
8                    Brackium Emendo               0   
9                      Cistem Aperio               0   
10                     Locking Spell               0   
11                    Blasting Curse               2   
12                   Cruciatus Curse               0   
13                    Severing Charm               1   
14                        Dissendium               0   
15                 Engorgement Charm               0   
16                           Episkey            

In [27]:
# show me all the spells with dominant topic 0 and 4
print(df[df["Dominant Topic"] == 0][["Spell Name", "Dominant Topic", "Topic Label"]])


                          Spell Name  Dominant Topic         Topic Label
0                    Summoning Charm               0  Combat/Force Magic
2   Launch an object up into the air               0  Combat/Force Magic
3                    Unlocking Charm               0  Combat/Force Magic
6                      Killing Curse               0  Combat/Force Magic
8                    Brackium Emendo               0  Combat/Force Magic
9                      Cistem Aperio               0  Combat/Force Magic
10                     Locking Spell               0  Combat/Force Magic
12                   Cruciatus Curse               0  Combat/Force Magic
14                        Dissendium               0  Combat/Force Magic
15                 Engorgement Charm               0  Combat/Force Magic
16                           Episkey               0  Combat/Force Magic
18                   Disarming Charm               0  Combat/Force Magic
19                     Expulso Curse               

In [28]:
print(df[df["Dominant Topic"] == 3][["Spell Name", "Dominant Topic", "Topic Label"]])

               Spell Name  Dominant Topic Topic Label
1      Water-Making Spell               3         NaN
17         Patronus Charm               3         NaN
26      Fire-Making Spell               3         NaN
43           Shield Charm               3         NaN
45        Protego totalum               3         NaN
47         Revulsion Jinx               3         NaN
56    Snake Summons Spell               3         NaN
59  Snake-Vanishing Spell               3         NaN
